In [ ]:
import pandas as pd

df = pd.read_csv("/content/소상공인시장진흥공단_상가(상권)정보_경기_202512.csv", encoding='utf-8', low_memory=False)
print(df.shape)
print(df.columns.tolist())

(650081, 39)
['상가업소번호', '상호명', '지점명', '상권업종대분류코드', '상권업종대분류명', '상권업종중분류코드', '상권업종중분류명', '상권업종소분류코드', '상권업종소분류명', '표준산업분류코드', '표준산업분류명', '시도코드', '시도명', '시군구코드', '시군구명', '행정동코드', '행정동명', '법정동코드', '법정동명', '지번코드', '대지구분코드', '대지구분명', '지번본번지', '지번부번지', '지번주소', '도로명코드', '도로명', '건물본번지', '건물부번지', '건물관리번호', '건물명', '도로명주소', '구우편번호', '신우편번호', '동정보', '층정보', '호정보', '경도', '위도']


In [ ]:
# 고양시 편의점만 추출
goyang = df[
    (df['시도명'] == '경기도') &
    (df['시군구명'] == '고양시')
]

convenience = goyang[
    goyang['상권업종소분류명'] == '편의점'
].copy()

print(f"고양시 편의점 수: {len(convenience)}개")
print(convenience['상호명'].value_counts().head(20))

고양시 편의점 수: 0개
Series([], Name: count, dtype: int64)


In [ ]:
# 편의점 관련 업종명 전체 확인
print(goyang['상권업종소분류명'].unique())
print("---")
print(f"고양시 전체 상가 수: {len(goyang)}개")

[]
---
고양시 전체 상가 수: 0개


In [ ]:
# 경기도 내 시군구명 전체 확인
gyeonggi = df[df['시도명'] == '경기도']
print(gyeonggi['시군구명'].unique())

['화성시' '성남시 분당구' '파주시' '수원시 장안구' '하남시' '안산시 단원구' '고양시 덕양구' '부천시 원미구' '광주시'
 '양평군' '성남시 수정구' '안양시 동안구' '김포시' '고양시 일산동구' '용인시 기흥구' '평택시' '수원시 팔달구'
 '안산시 상록구' '부천시 소사구' '양주시' '안성시' '시흥시' '이천시' '용인시 수지구' '의정부시' '수원시 영통구'
 '여주시' '포천시' '부천시 오정구' '군포시' '남양주시' '동두천시' '의왕시' '성남시 중원구' '수원시 권선구' '구리시'
 '용인시 처인구' '고양시 일산서구' '안양시 만안구' '연천군' '오산시' '광명시' '가평군' '과천시']


In [ ]:
# 고양시 3개 구 모두 포함
goyang = df[df['시군구명'].str.contains('고양시')]

convenience = goyang[
    goyang['상권업종소분류명'].str.contains('편의점', na=False)
].copy()

print(f"고양시 편의점 수: {len(convenience)}개")
print(convenience['상호명'].value_counts().head(20))

고양시 편의점 수: 953개
상호명
씨유일산       13
지에스25일산     9
세븐일레븐일산     7
씨유고양        6
씨유킨텍스       6
씨유          6
지에스25삼송     5
세븐일레븐       5
지에스25고양     5
이마트24       4
씨유행신        4
이마트24일산     4
지에스25탄현     4
지에스25백석     4
지에스25덕양     3
씨유대화        3
씨유정발산       3
지에스25대화     3
씨유원흥        3
이마트24고양     3
Name: count, dtype: int64


In [ ]:
# 브랜드 분류
def classify_brand(name):
    name = str(name).upper()
    if '씨유' in name or 'CU' in name:
        return 'CU'
    elif '지에스' in name or 'GS' in name:
        return 'GS25'
    elif '세븐' in name or '7-' in name:
        return '세븐일레븐'
    elif '이마트' in name:
        return '이마트24'
    else:
        return '기타'

convenience['브랜드'] = convenience['상호명'].apply(classify_brand)

# 좌표 정제 (고양시 bbox)
convenience = convenience.dropna(subset=['경도', '위도'])
convenience = convenience[
    (convenience['위도'].between(37.55, 37.75)) &
    (convenience['경도'].between(126.70, 126.95))
]

# 필요한 컬럼만 추출
result = convenience[['상호명', '브랜드', '도로명주소', '경도', '위도', '행정동명']].copy()
result.columns = ['name', 'brand', 'address', 'lng', 'lat', 'dong']

print(f"최종 편의점 수: {len(result)}개")
print(result['brand'].value_counts())

# CSV 저장
result.to_csv('goyang_convenience_stores.csv', index=False, encoding='utf-8-sig')
print("\n저장 완료!")
print(result.head())

최종 편의점 수: 951개
brand
CU       290
GS25     288
세븐일레븐    217
이마트24     78
기타        78
Name: count, dtype: int64

저장 완료!
             name  brand                      address         lng        lat  \
597    씨유일산윈스턴파크점     CU          경기도 고양시 일산동구 강송로 41  126.791399  37.644550   
779       CU마두건영점     CU        경기도 고양시 일산동구 산두로 21-2  126.789132  37.661034   
1552   지에스25웨스턴돔점   GS25         경기도 고양시 일산동구 정발산로 24  126.772046  37.655903   
1603  이마트24행주나루터점  이마트24  경기도 고양시 덕양구 행주산성로144번길 77-2  126.815004  37.602478   
3908      씨유고양내유점     CU           경기도 고양시 덕양구 유산길 22  126.845471  37.726163   

      dong  
597   백석1동  
779   마두1동  
1552  장항2동  
1603   행주동  
3908   관산동  


In [ ]:
!pip install folium -q

import folium
from folium.plugins import HeatMap

# 브랜드별 색상
brand_colors = {
    'CU':     '#00a0e9',
    'GS25':   '#009d4e',
    '세븐일레븐': '#ff6600',
    '이마트24':  '#e11400',
    '기타':    '#888888'
}

# 지도 초기화
m = folium.Map(location=[37.66, 126.83], zoom_start=12, tiles='CartoDB positron')

# 편의점 마커
for _, row in result.iterrows():
    color = brand_colors.get(row['brand'], '#888888')
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        tooltip=f"{row['name']} ({row['brand']})",
        popup=folium.Popup(f"<b>{row['name']}</b><br>{row['brand']}<br>{row['address']}", max_width=200)
    ).add_to(m)

# 범례
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:12px 16px;border-radius:10px;
     border:1px solid #ddd;font-size:13px;line-height:2;
     box-shadow:0 2px 8px rgba(0,0,0,0.15)">
  <b>고양시 편의점 현황</b><br>
  <span style="color:#00a0e9">●</span> CU (290)<br>
  <span style="color:#009d4e">●</span> GS25 (288)<br>
  <span style="color:#ff6600">●</span> 세븐일레븐 (217)<br>
  <span style="color:#e11400">●</span> 이마트24 (78)
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# 저장
m.save('goyang_map.html')
print("저장 완료! goyang_map.html 파일을 다운받아서 브라우저로 열어보세요.")

저장 완료! goyang_map.html 파일을 다운받아서 브라우저로 열어보세요.


In [ ]:
!pip install folium scikit-learn -q

import pandas as pd
import numpy as np
import folium
from sklearn.cluster import KMeans

# ── 1. 저장된 데이터 로드 ─────────────────────────────────────────────────────
result = pd.read_csv('goyang_convenience_stores.csv')

# ── 2. 고양시 전체를 격자로 나누기 ───────────────────────────────────────────
# 고양시 bbox: 위도 37.57~37.74, 경도 126.72~126.96
# 500m 간격 격자 생성 (약 0.0045도 = 500m)
lat_steps = np.arange(37.57, 37.74, 0.0045)
lng_steps = np.arange(126.72, 126.96, 0.0045)

grid_points = []
for lat in lat_steps:
    for lng in lng_steps:
        grid_points.append({'lat': lat, 'lng': lng})

grid = pd.DataFrame(grid_points)
print(f"격자 포인트 수: {len(grid)}개")

# ── 3. 각 격자점에서 가장 가까운 편의점까지 거리 계산 ─────────────────────────
from scipy.spatial.distance import cdist

store_coords = result[['lat', 'lng']].values
grid_coords  = grid[['lat', 'lng']].values

# 거리 계산 (도 → 미터, 위도 1도 ≈ 111km, 경도 1도 ≈ 88km in 고양시)
def dist_meters(a, b):
    dlat = (a[:, 0:1] - b[:, 0]) * 111000
    dlng = (a[:, 1:2] - b[:, 1]) * 88000
    return np.sqrt(dlat**2 + dlng**2)

distances = dist_meters(grid_coords, store_coords)
grid['nearest_store_m'] = distances.min(axis=1).round()

# ── 4. 공백 지역 추출 (편의점 500m 이상 없는 격자점) ─────────────────────────
gap = grid[grid['nearest_store_m'] >= 500].copy()
print(f"공백 격자 수: {len(gap)}개")

# ── 5. 공백 지역 클러스터링 (인접한 공백끼리 묶기) ───────────────────────────
km = KMeans(n_clusters=15, random_state=42, n_init=10)
gap['cluster'] = km.fit_predict(gap[['lat', 'lng']])

# 클러스터별 중심 + 평균 거리 계산
centroids = gap.groupby('cluster').agg(
    lat=('lat', 'mean'),
    lng=('lng', 'mean'),
    avg_gap_m=('nearest_store_m', 'mean'),
    size=('lat', 'count')
).reset_index()

# 공백이 클수록, 면적이 넓을수록 우선순위 높음
centroids['score'] = (centroids['avg_gap_m'] * centroids['size']).round()
centroids = centroids.sort_values('score', ascending=False).reset_index(drop=True)
centroids['rank'] = centroids.index + 1

print("\n추천 부지 TOP 10:")
print(centroids[['rank','lat','lng','avg_gap_m','size','score']].head(10).to_string())

격자 포인트 수: 2052개
공백 격자 수: 1373개

추천 부지 TOP 10:
   rank        lat         lng    avg_gap_m  size     score
0     1  37.589741  126.741651  5617.915094   106  595499.0
1     2  37.586975  126.935287  3935.881188   101  397524.0
2     3  37.596539  126.788209  2447.551181   127  310839.0
3     4  37.631765  126.936159  2232.150442   113  252233.0
4     5  37.718887  126.937984  2504.268817    93  232897.0
5     6  37.635500  126.740625  1978.981481   108  213730.0
6     7  37.680250  126.930693  1638.613208   106  173693.0
7     8  37.584950  126.838400  1595.933333    90  143634.0
8     9  37.713270  126.734047  1919.648649    74  142054.0
9    10  37.603188  126.891216  1095.932692   104  113977.0


In [ ]:
# ── 6. 지도 시각화 ────────────────────────────────────────────────────────────
m = folium.Map(location=[37.66, 126.83], zoom_start=12, tiles='CartoDB positron')

# 기존 편의점 마커
brand_colors = {
    'CU':     '#00a0e9',
    'GS25':   '#009d4e',
    '세븐일레븐': '#ff6600',
    '이마트24':  '#e11400',
    '기타':    '#888888'
}

store_group = folium.FeatureGroup(name='기존 편의점', show=True)
for _, row in result.iterrows():
    color = brand_colors.get(row['brand'], '#888888')
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius=4,
        color=color, fill=True, fill_color=color, fill_opacity=0.8,
        tooltip=f"{row['name']} ({row['brand']})"
    ).add_to(store_group)
store_group.add_to(m)

# 공백 히트맵
from folium.plugins import HeatMap
heat_group = folium.FeatureGroup(name='공백 밀도 히트맵', show=True)
heat_data = [[r['lat'], r['lng'], r['nearest_store_m']] for _, r in gap.iterrows()]
HeatMap(heat_data, radius=25, blur=20,
        gradient={0.4:'#ffffb2', 0.6:'#fd8d3c', 0.8:'#f03b20', 1.0:'#bd0026'}
).add_to(heat_group)
heat_group.add_to(m)

# 추천 부지 TOP 10 마커
top10 = centroids.head(10)
rec_group = folium.FeatureGroup(name='추천 부지 TOP 10', show=True)
for _, row in top10.iterrows():
    # 반경 원
    folium.Circle(
        location=[row['lat'], row['lng']],
        radius=500,
        color='#c084fc', fill=True, fill_color='#c084fc', fill_opacity=0.12, weight=2
    ).add_to(rec_group)
    # 마커
    folium.Marker(
        location=[row['lat'], row['lng']],
        icon=folium.DivIcon(html=f"""
            <div style="
                width:28px;height:28px;border-radius:50%;
                background:#7c3aed;border:2px solid white;
                display:flex;align-items:center;justify-content:center;
                font-size:12px;font-weight:700;color:white;
                box-shadow:0 2px 6px rgba(0,0,0,0.3);
                line-height:28px;text-align:center;">
                {int(row['rank'])}
            </div>""",
            icon_size=(28, 28), icon_anchor=(14, 14)
        ),
        tooltip=f"추천 #{int(row['rank'])} | 평균 공백 {int(row['avg_gap_m'])}m",
        popup=folium.Popup(
            f"<b>추천 부지 #{int(row['rank'])}</b><br>"
            f"평균 공백 거리: {int(row['avg_gap_m'])}m<br>"
            f"공백 면적 크기: {int(row['size'])} 격자<br>"
            f"우선순위 점수: {int(row['score'])}",
            max_width=220
        )
    ).add_to(rec_group)
rec_group.add_to(m)

# 범례
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:12px 16px;border-radius:10px;
     border:1px solid #ddd;font-size:12px;line-height:2;
     box-shadow:0 2px 8px rgba(0,0,0,0.15)">
  <b style="font-size:13px">고양시 편의점 공백 분석</b><br>
  <span style="color:#00a0e9">●</span> CU &nbsp;
  <span style="color:#009d4e">●</span> GS25 &nbsp;
  <span style="color:#ff6600">●</span> 세븐일레븐 &nbsp;
  <span style="color:#e11400">●</span> 이마트24<br>
  <span style="color:#fd8d3c">■</span> 공백 밀도 히트맵<br>
  <span style="color:#7c3aed">●</span> 추천 부지 TOP 10
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=False).add_to(m)

m.save('goyang_gap_map.html')
print("저장 완료! goyang_gap_map.html 다운받아서 열어보세요.")

저장 완료! goyang_gap_map.html 다운받아서 열어보세요.


In [ ]:
# 추천 부지 TOP 10 Google Maps 링크 출력
print("=== 추천 부지 TOP 10 위치 확인 ===\n")
for _, row in centroids.head(10).iterrows():
    maps_url = f"https://maps.google.com/?q={row['lat']},{row['lng']}"
    print(f"#{int(row['rank'])} | 평균공백 {int(row['avg_gap_m'])}m | {maps_url}")

=== 추천 부지 TOP 10 위치 확인 ===

#1 | 평균공백 5617m | https://maps.google.com/?q=37.58974056603774,126.7416509433962
#2 | 평균공백 3935m | https://maps.google.com/?q=37.586975247524755,126.93528712871255
#3 | 평균공백 2447m | https://maps.google.com/?q=37.596539370078744,126.78820866141722
#4 | 평균공백 2232m | https://maps.google.com/?q=37.63176548672567,126.93615929203506
#5 | 평균공백 2504m | https://maps.google.com/?q=37.7188870967742,126.93798387096741
#6 | 평균공백 1978m | https://maps.google.com/?q=37.63550000000001,126.74062499999997
#7 | 평균공백 1638m | https://maps.google.com/?q=37.68025,126.93069339622609
#8 | 평균공백 1595m | https://maps.google.com/?q=37.58495,126.83839999999982
#9 | 평균공백 1919m | https://maps.google.com/?q=37.71327027027028,126.73404729729727
#10 | 평균공백 1095m | https://maps.google.com/?q=37.603187500000004,126.89121634615358


In [ ]:
# 각 추천 부지 좌표를 역지오코딩으로 주소 확인 + 산/공원 여부 판단
# nominatim 무료 API 활용

!pip install geopy -q
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

geolocator = Nominatim(user_agent="goyang_analysis")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

print("=== 추천 부지 TOP 10 주소 확인 ===\n")
results = []
for _, row in centroids.head(10).iterrows():
    location = reverse(f"{row['lat']}, {row['lng']}", language='ko')
    addr = location.address if location else "주소 없음"
    results.append({
        'rank': int(row['rank']),
        'lat': row['lat'],
        'lng': row['lng'],
        'avg_gap_m': int(row['avg_gap_m']),
        'score': int(row['score']),
        'address': addr
    })
    print(f"#{int(row['rank'])} | {int(row['avg_gap_m'])}m | {addr}")
    time.sleep(1)

=== 추천 부지 TOP 10 주소 확인 ===

#1 | 5617m | 계양강화고속도로, 검단신도시 로제비앙 라포레, 계양1동, 계양구, 인천광역시, 경기도, 21004, 대한민국
#2 | 3935m | 홍제천 자전거길, 홍은동, 홍은2동, 서대문구, 서울특별시, 03653, 대한민국
#3 | 2447m | 아라김포여객터미널, 74, 아라육로270번길, 고촌읍, 김포시, 10135, 대한민국
#4 | 2232m | [북한산둘레길] 8구간 구름정원길, 진관동, 은평구, 서울특별시, 03309, 대한민국
#5 | 2504m | 신호국로, 장흥면, 양주시, 11520, 대한민국
#6 | 1978m | 금포로, 고촌읍, 김포시, 경기도, 10111, 대한민국
#7 | 1638m | 독재동추사필적암각문, 수도권제1순환고속도로, 장흥면, 양주시, 경기도, 11525, 대한민국
#8 | 1595m | 마곡대교, 한강 자전거길, 마곡지구도시개발지구, 가양1동, 강서구, 서울특별시, 07521, 대한민국
#9 | 1919m | 운정신도시아이파크아파트, 운정3동, 파주시, 10903, 대한민국
#10 | 1095m | 26, 꽃내음2길, 향동동, 화전동, 덕양구, 고양시, 10545, 대한민국


In [ ]:
# 고양시 정확한 경계로 bbox 재설정
# 덕양구 + 일산동구 + 일산서구 실제 범위
# 위도: 37.57 ~ 37.73, 경도: 126.74 ~ 126.97

lat_steps = np.arange(37.57, 37.73, 0.0045)
lng_steps = np.arange(126.74, 126.97, 0.0045)

grid_points = []
for lat in lat_steps:
    for lng in lng_steps:
        grid_points.append({'lat': lat, 'lng': lng})

grid = pd.DataFrame(grid_points)
print(f"격자 포인트 수: {len(grid)}개")

# 편의점까지 거리 재계산
store_coords = result[['lat', 'lng']].values
grid_coords  = grid[['lat', 'lng']].values

distances = dist_meters(grid_coords, store_coords)
grid['nearest_store_m'] = distances.min(axis=1).round()

# 공백 지역 추출
gap = grid[grid['nearest_store_m'] >= 500].copy()
print(f"공백 격자 수: {len(gap)}개")

# 클러스터링
km = KMeans(n_clusters=15, random_state=42, n_init=10)
gap['cluster'] = km.fit_predict(gap[['lat', 'lng']])

centroids = gap.groupby('cluster').agg(
    lat=('lat', 'mean'),
    lng=('lng', 'mean'),
    avg_gap_m=('nearest_store_m', 'mean'),
    size=('lat', 'count')
).reset_index()

centroids['score'] = (centroids['avg_gap_m'] * centroids['size']).round()
centroids = centroids.sort_values('score', ascending=False).reset_index(drop=True)
centroids['rank'] = centroids.index + 1

# 역지오코딩으로 주소 확인
print("\n=== 추천 부지 TOP 10 재확인 ===\n")
res2 = []
for _, row in centroids.head(10).iterrows():
    location = reverse(f"{row['lat']}, {row['lng']}", language='ko')
    addr = location.address if location else "주소 없음"
    res2.append({
        'rank': int(row['rank']),
        'lat': round(row['lat'], 6),
        'lng': round(row['lng'], 6),
        'avg_gap_m': int(row['avg_gap_m']),
        'score': int(row['score']),
        'address': addr
    })
    print(f"#{int(row['rank'])} | {int(row['avg_gap_m'])}m | {addr}")
    time.sleep(1)

격자 포인트 수: 1872개
공백 격자 수: 1216개

=== 추천 부지 TOP 10 재확인 ===

#1 | 4869m | 세무서10길, 홍제3동, 서대문구, 서울특별시, 03622, 대한민국
#2 | 4796m | 인향로, 계양1동, 고촌읍, 김포시, 경기도, 10130, 대한민국
#3 | 2557m | 비법정탐방로, 진관동, 은평구, 서울특별시, 03308, 대한민국
#4 | 2796m | 석현천 자전거길, 장흥면, 양주시, 11520, 대한민국
#5 | 2121m | 방화2동, 강서구, 서울특별시, 07502, 대한민국
#6 | 2289m | 북한산로, 효자동, 덕양구, 고양시, 10582, 대한민국
#7 | 1880m | 증산중학교, 증산로5길, 증산동, 은평구, 서울특별시, 03493, 대한민국
#8 | 1654m | 금포로, 고촌읍, 김포시, 경기도, 10128, 대한민국
#9 | 1445m | 한강북자전거길, 대덕동, 덕양구, 고양시, 경기도, 07521, 대한민국
#10 | 1115m | 용두로, 창릉동, 덕양구, 고양시, 10549, 대한민국


In [ ]:
# 행정동별 편의점 수 분석
dong_counts = result.groupby('dong').agg(
    store_count=('name', 'count')
).reset_index()

# 브랜드별 분포도 추가
brand_pivot = result.groupby(['dong', 'brand']).size().unstack(fill_value=0).reset_index()
dong_stats = dong_counts.merge(brand_pivot, on='dong', how='left')

# 편의점 수 기준 정렬
dong_stats = dong_stats.sort_values('store_count', ascending=False)

print("=== 행정동별 편의점 현황 ===")
print(f"\n편의점 많은 TOP 10:")
print(dong_stats.head(10)[['dong','store_count']].to_string(index=False))

print(f"\n편의점 적은 하위 10개 동:")
print(dong_stats.tail(10)[['dong','store_count']].to_string(index=False))

print(f"\n전체 행정동 수: {len(dong_stats)}개")
print(f"평균 편의점 수: {dong_stats['store_count'].mean():.1f}개/동")
print(f"중앙값: {dong_stats['store_count'].median():.0f}개/동")

=== 행정동별 편의점 현황 ===

편의점 많은 TOP 10:
dong  store_count
장항2동           64
 고봉동           47
 흥도동           43
 대화동           38
백석1동           36
 송포동           36
 관산동           34
정발산동           33
 식사동           31
성사1동           30

편의점 적은 하위 10개 동:
dong  store_count
 주교동           13
 능곡동           13
일산1동           13
중산2동           12
행신1동           11
삼송2동           10
행신3동            9
마두2동            8
탄현2동            5
성사2동            4

전체 행정동 수: 44개
평균 편의점 수: 21.6개/동
중앙값: 18개/동


In [ ]:
# 고봉동 편의점 샘플 확인
print("=== 고봉동 편의점 목록 ===")
print(result[result['dong'] == '고봉동'][['name','brand','address']].to_string())

=== 고봉동 편의점 목록 ===
                   name  brand                     address
10           세븐일레븐일산지영점  세븐일레븐    경기도 고양시 일산동구 공릉천로355번길 1
17            세븐일산설문대로점  세븐일레븐    경기도 고양시 일산동구 고봉로678번길 13
21            GS25고양성석점   GS25        경기도 고양시 일산동구 성현로 201
29                지에스마트   GS25        경기도 고양시 일산동구 고봉로 684
41            인스케어쌍용플러스     기타        경기도 고양시 일산동구 성현로 615
65              씨유일산운정로     CU        경기도 고양시 일산동구 운정로 190
72          세븐일레븐일산지영로점  세븐일레븐         경기도 고양시 일산동구 지영로 52
111             지에스25성석   GS25        경기도 고양시 일산동구 성석로 331
150               씨유사리현     CU        경기도 고양시 일산동구 공릉천로 70
165         지에스25고양문봉길점   GS25         경기도 고양시 일산동구 문봉길 43
181           씨유일산사리현로점     CU     경기도 고양시 일산동구 사리현로 207-1
222          이마트24일산고봉점  이마트24      경기도 고양시 일산동구 성현로 397-6
227          씨유동양인재개발원점     CU  경기도 고양시 일산동구 성현로268번길 68-7
244          세븐일레븐일산문봉점  세븐일레븐       경기도 고양시 일산동구 견달산로 359
274        지에스25고봉로장고개점   GS25     경기도 고양시 일산동구 고봉로770번길 5
300         세븐일레븐일산은마길점  세븐일레븐       

In [ ]:
# 구 기준으로 재집계 (주소에서 추출)
result['gu'] = result['address'].str.extract(r'고양시\s+(덕양구|일산동구|일산서구)')

# 구 + 행정동 조합으로 정확한 집계
# 도로명주소에서 동 이름 추출
result['dong_clean'] = result['address'].str.extract(r'((?:행신|화정|능곡|삼송|원당|성사|주교|덕양|흥도|효자|관산|고봉|벽제|대장|강매|화전|향동|창릉|대덕|현천|동산|내곡|신원|일산|탄현|마두|정발산|백석|중산|대화|장항|풍산|송포|식사|성석|지영|사리현|설문|문봉|지축|구산|주엽|대화|화정)\d*동)')

gu_dong = result.groupby(['gu', 'dong_clean']).agg(
    store_count=('name', 'count')
).reset_index().dropna()

gu_dong = gu_dong.sort_values('store_count', ascending=False)

print("=== 구별 편의점 현황 ===")
print(result['gu'].value_counts())

print("\n=== 동별 편의점 TOP 15 ===")
print(gu_dong.head(15).to_string(index=False))

print("\n=== 편의점 적은 동 하위 15 ===")
print(gu_dong.tail(15).to_string(index=False))

=== 구별 편의점 현황 ===
gu
덕양구     406
일산동구    322
일산서구    223
Name: count, dtype: int64

=== 동별 편의점 TOP 15 ===
  gu dong_clean  store_count
일산동구        일산동          322

=== 편의점 적은 동 하위 15 ===
  gu dong_clean  store_count
일산동구        일산동          322


In [ ]:
# 도로명주소에서 동 이름 추출 (시군구 뒤에 오는 동/읍/면)
result['dong_clean'] = result['address'].str.extract(r'(?:덕양구|일산동구|일산서구)\s+(\S+동|\S+읍|\S+면)')

print("추출된 동 샘플:")
print(result[['name','address','dong_clean']].head(20).to_string())

print("\n고유 동 목록:")
print(sorted(result['dong_clean'].dropna().unique()))

추출된 동 샘플:
             name                      address dong_clean
0      씨유일산윈스턴파크점          경기도 고양시 일산동구 강송로 41        NaN
1         CU마두건영점        경기도 고양시 일산동구 산두로 21-2        NaN
2      지에스25웨스턴돔점         경기도 고양시 일산동구 정발산로 24        NaN
3     이마트24행주나루터점  경기도 고양시 덕양구 행주산성로144번길 77-2        NaN
4         씨유고양내유점           경기도 고양시 덕양구 유산길 22        NaN
5       GS25일산제니스       경기도 고양시 일산서구 일현로 97-11        NaN
6            씨유원흥         경기도 고양시 덕양구 권율대로 512        NaN
7     GS25장항세이프빌점      경기도 고양시 일산동구 무궁화로 43-15        NaN
8       세븐일레븐서오릉점       경기도 고양시 덕양구 서오릉로 396-5        NaN
9   세븐일레븐일산탄현1단지점          경기도 고양시 일산서구 일현로 70        NaN
10     세븐일레븐일산지영점     경기도 고양시 일산동구 공릉천로355번길 1        NaN
11         씨유하이파크       경기도 고양시 일산서구 하이파크3로 61        NaN
12       씨유일산마두역점        경기도 고양시 일산동구 중앙로 1187        NaN
13          업소명없음         경기도 고양시 덕양구 권율대로 690        NaN
14        세븐일레븐고양           경기도 고양시 덕양구 원흥5로 4        NaN
15     세븐일산후곡마을점주         경기도 고양시 일산서구 일산로 535        NaN
16  

In [ ]:
# 원본 데이터에서 행정동명 다시 가져오기
goyang = df[df['시군구명'].str.contains('고양시')]
convenience2 = goyang[goyang['상권업종소분류명'].str.contains('편의점', na=False)].copy()
convenience2['브랜드'] = convenience2['상호명'].apply(classify_brand)

# 좌표 정제
convenience2 = convenience2.dropna(subset=['경도', '위도'])
convenience2 = convenience2[
    (convenience2['위도'].between(37.57, 37.73)) &
    (convenience2['경도'].between(126.74, 126.97))
]

# 행정동명 공백 제거
convenience2['행정동명'] = convenience2['행정동명'].str.strip()
convenience2['시군구명'] = convenience2['시군구명'].str.strip()

# 구 + 동 기준 집계
dong_stats = convenience2.groupby(['시군구명', '행정동명']).agg(
    store_count=('상호명', 'count')
).reset_index()

dong_stats = dong_stats.sort_values('store_count', ascending=False).reset_index(drop=True)
dong_stats['rank'] = dong_stats.index + 1

print(f"전체 동 수: {len(dong_stats)}개")
print(f"평균 편의점 수: {dong_stats['store_count'].mean():.1f}개/동")
print()
print("=== 편의점 많은 TOP 10 ===")
print(dong_stats.head(10)[['시군구명','행정동명','store_count']].to_string(index=False))
print()
print("=== 편의점 적은 하위 10 ===")
print(dong_stats.tail(10)[['시군구명','행정동명','store_count']].to_string(index=False))

전체 동 수: 43개
평균 편의점 수: 21.4개/동

=== 편의점 많은 TOP 10 ===
    시군구명 행정동명  store_count
고양시 일산동구 장항2동           64
고양시 일산동구  고봉동           47
 고양시 덕양구  흥도동           43
고양시 일산서구  대화동           38
고양시 일산동구 백석1동           36
 고양시 덕양구  관산동           34
고양시 일산동구 정발산동           33
고양시 일산동구  식사동           31
 고양시 덕양구 성사1동           30
 고양시 덕양구 행신2동           27

=== 편의점 적은 하위 10 ===
    시군구명 행정동명  store_count
 고양시 덕양구  주교동           13
 고양시 덕양구  능곡동           13
고양시 일산서구 일산1동           13
고양시 일산동구 중산2동           12
 고양시 덕양구 행신1동           11
 고양시 덕양구 삼송2동           10
 고양시 덕양구 행신3동            9
고양시 일산동구 마두2동            8
고양시 일산서구 탄현2동            5
 고양시 덕양구 성사2동            4


In [ ]:
from scipy.spatial.distance import cdist
import numpy as np

# 동별 편의점 좌표 + 평균 최근접 거리 계산
dong_analysis = []

for (gu, dong), group in convenience2.groupby(['시군구명', '행정동명']):
    coords = group[['위도', '경도']].values
    n = len(coords)

    if n >= 2:
        # 자기 자신 제외한 최근접 거리
        dist = cdist(coords, coords)
        np.fill_diagonal(dist, np.inf)
        min_dists = dist.min(axis=1)
        # 도 → 미터 변환
        avg_nn_m = (min_dists.mean() * 111000).round()
    else:
        avg_nn_m = 9999  # 1개면 고립

    dong_analysis.append({
        'gu': gu.strip(),
        'dong': dong.strip(),
        'store_count': n,
        'avg_nn_m': int(avg_nn_m)  # 편의점 간 평균 최근접 거리
    })

da = pd.DataFrame(dong_analysis)

# 우선순위 점수: 거리 멀수록 + 편의점 적을수록 높음
da['score'] = (da['avg_nn_m'] * (1 / da['store_count'])).round(1)
da = da.sort_values('score', ascending=False).reset_index(drop=True)
da['rank'] = da.index + 1

print("=== 편의점 공백 심각도 TOP 15 (거리 기반) ===")
print(da.head(15)[['rank','gu','dong','store_count','avg_nn_m']].to_string(index=False))

=== 편의점 공백 심각도 TOP 15 (거리 기반) ===
 rank       gu dong  store_count  avg_nn_m
    1 고양시 일산서구 탄현2동            5       262
    2  고양시 덕양구 성사2동            4       202
    3  고양시 덕양구  능곡동           13       473
    4  고양시 덕양구  원신동           14       462
    5  고양시 덕양구 행신3동            9       176
    6 고양시 일산서구  덕이동           15       285
    7  고양시 덕양구  효자동           18       340
    8 고양시 일산동구 중산2동           12       203
    9 고양시 일산동구 장항1동           16       250
   10  고양시 덕양구  주교동           13       189
   11  고양시 덕양구  대덕동           16       231
   12  고양시 덕양구 삼송2동           10       132
   13  고양시 덕양구  창릉동           18       225
   14 고양시 일산서구 일산1동           13       159
   15 고양시 일산동구  풍산동           20       234


In [ ]:
# 동별 중심 좌표 계산
dong_centers = convenience2.groupby(['시군구명', '행정동명']).agg(
    center_lat=('위도', 'mean'),
    center_lng=('경도', 'mean')
).reset_index()
dong_centers['gu'] = dong_centers['시군구명'].str.strip()
dong_centers['dong'] = dong_centers['행정동명'].str.strip()

da = da.merge(dong_centers[['gu','dong','center_lat','center_lng']], on=['gu','dong'], how='left')

# 최종 지도
m = folium.Map(location=[37.66, 126.83], zoom_start=12, tiles='CartoDB positron')

# 기존 편의점
store_group = folium.FeatureGroup(name='기존 편의점', show=True)
brand_colors = {'CU':'#00a0e9','GS25':'#009d4e','세븐일레븐':'#ff6600','이마트24':'#e11400','기타':'#aaaaaa'}
for _, row in convenience2.iterrows():
    color = brand_colors.get(row['브랜드'], '#aaaaaa')
    folium.CircleMarker(
        location=[row['위도'], row['경도']],
        radius=4, color=color, fill=True, fill_color=color, fill_opacity=0.7,
        tooltip=f"{row['상호명']} ({row['브랜드']})"
    ).add_to(store_group)
store_group.add_to(m)

# 공백 심각도 TOP 10 마커
rec_group = folium.FeatureGroup(name='공백 심각 TOP 10', show=True)
top10 = da.head(10).dropna(subset=['center_lat','center_lng'])

rank_colors = {1:'#7c3aed', 2:'#7c3aed', 3:'#dc2626', 4:'#dc2626', 5:'#ea580c',
               6:'#ea580c', 7:'#ca8a04', 8:'#ca8a04', 9:'#16a34a', 10:'#16a34a'}

for _, row in top10.iterrows():
    color = rank_colors.get(int(row['rank']), '#888888')
    folium.Circle(
        location=[row['center_lat'], row['center_lng']],
        radius=600, color=color, fill=True, fill_color=color, fill_opacity=0.12, weight=2
    ).add_to(rec_group)
    folium.Marker(
        location=[row['center_lat'], row['center_lng']],
        icon=folium.DivIcon(
            html=f"""<div style="
                width:30px;height:30px;border-radius:50%;
                background:{color};border:2px solid white;
                display:flex;align-items:center;justify-content:center;
                font-size:13px;font-weight:700;color:white;
                box-shadow:0 2px 6px rgba(0,0,0,0.35);
                line-height:30px;text-align:center;">{int(row['rank'])}</div>""",
            icon_size=(30,30), icon_anchor=(15,15)
        ),
        tooltip=f"#{int(row['rank'])} {row['dong']} | 편의점 {int(row['store_count'])}개 | 평균간격 {int(row['avg_nn_m'])}m",
        popup=folium.Popup(
            f"<b>공백 #{int(row['rank'])}: {row['dong']}</b><br>"
            f"구: {row['gu']}<br>"
            f"편의점 수: {int(row['store_count'])}개<br>"
            f"평균 최근접 거리: {int(row['avg_nn_m'])}m<br>"
            f"<a href='https://maps.google.com/?q={row['center_lat']},{row['center_lng']}' target='_blank'>구글 지도로 보기</a>",
            max_width=240
        )
    ).add_to(rec_group)
rec_group.add_to(m)

# 범례
legend_html = f"""
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:14px 18px;border-radius:10px;
     border:1px solid #ddd;font-size:12px;line-height:2;
     box-shadow:0 2px 8px rgba(0,0,0,0.15)">
  <b style="font-size:13px">고양시 편의점 공백 분석</b><br>
  <span style="color:#00a0e9">●</span> CU &nbsp;
  <span style="color:#009d4e">●</span> GS25 &nbsp;
  <span style="color:#ff6600">●</span> 세븐일레븐 &nbsp;
  <span style="color:#e11400">●</span> 이마트24<br>
  <span style="color:#7c3aed">●</span> 1~2위 &nbsp;
  <span style="color:#dc2626">●</span> 3~4위 &nbsp;
  <span style="color:#ea580c">●</span> 5~6위<br>
  <span style="color:#ca8a04">●</span> 7~8위 &nbsp;
  <span style="color:#16a34a">●</span> 9~10위
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False).add_to(m)

m.save('goyang_final_map.html')
print("완료! goyang_final_map.html 다운받아서 올려주세요!")

완료! goyang_final_map.html 다운받아서 올려주세요!


In [ ]:
# 위성 타일로 변경
m = folium.Map(location=[37.66, 126.83], zoom_start=12)

# 위성 타일 추가 (Google Satellite)
folium.TileLayer(
    tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
    attr='Google Satellite',
    name='위성지도',
    show=True
).add_to(m)

# 일반 지도도 선택 가능하게
folium.TileLayer('CartoDB positron', name='일반지도', show=False).add_to(m)

# 기존 편의점
store_group = folium.FeatureGroup(name='기존 편의점', show=True)
brand_colors = {'CU':'#00a0e9','GS25':'#00dd66','세븐일레븐':'#ff6600','이마트24':'#ff4444','기타':'#cccccc'}
for _, row in convenience2.iterrows():
    color = brand_colors.get(row['브랜드'], '#cccccc')
    folium.CircleMarker(
        location=[row['위도'], row['경도']],
        radius=4, color='white', weight=1,
        fill=True, fill_color=color, fill_opacity=0.9,
        tooltip=f"{row['상호명']} ({row['브랜드']})"
    ).add_to(store_group)
store_group.add_to(m)

# 공백 TOP 10
rec_group = folium.FeatureGroup(name='공백 심각 TOP 10', show=True)
rank_colors = {1:'#a855f7',2:'#a855f7',3:'#ef4444',4:'#ef4444',5:'#f97316',
               6:'#f97316',7:'#eab308',8:'#eab308',9:'#22c55e',10:'#22c55e'}

for _, row in top10.iterrows():
    color = rank_colors.get(int(row['rank']), '#888888')
    folium.Circle(
        location=[row['center_lat'], row['center_lng']],
        radius=600, color=color, fill=True, fill_color=color,
        fill_opacity=0.2, weight=2.5
    ).add_to(rec_group)
    folium.Marker(
        location=[row['center_lat'], row['center_lng']],
        icon=folium.DivIcon(
            html=f"""<div style="
                width:32px;height:32px;border-radius:50%;
                background:{color};border:2.5px solid white;
                display:flex;align-items:center;justify-content:center;
                font-size:13px;font-weight:700;color:white;
                box-shadow:0 2px 8px rgba(0,0,0,0.5);
                line-height:32px;text-align:center;">{int(row['rank'])}</div>""",
            icon_size=(32,32), icon_anchor=(16,16)
        ),
        tooltip=f"#{int(row['rank'])} {row['dong']} | 편의점 {int(row['store_count'])}개 | 평균간격 {int(row['avg_nn_m'])}m",
        popup=folium.Popup(
            f"<b>공백 #{int(row['rank'])}: {row['dong']}</b><br>"
            f"구: {row['gu']}<br>"
            f"편의점 수: {int(row['store_count'])}개<br>"
            f"평균 최근접 거리: {int(row['avg_nn_m'])}m<br>"
            f"<a href='https://maps.google.com/?q={row['center_lat']},{row['center_lng']}' target='_blank'>구글 지도로 보기</a>",
            max_width=240
        )
    ).add_to(rec_group)
rec_group.add_to(m)

# 범례
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:rgba(0,0,0,0.75);padding:14px 18px;border-radius:10px;
     font-size:12px;line-height:2;color:white;">
  <b style="font-size:13px">고양시 편의점 공백 분석</b><br>
  <span style="color:#00a0e9">●</span> CU &nbsp;
  <span style="color:#00dd66">●</span> GS25 &nbsp;
  <span style="color:#ff6600">●</span> 세븐일레븐 &nbsp;
  <span style="color:#ff4444">●</span> 이마트24<br>
  <span style="color:#a855f7">●</span> 1~2위 &nbsp;
  <span style="color:#ef4444">●</span> 3~4위 &nbsp;
  <span style="color:#f97316">●</span> 5~6위<br>
  <span style="color:#eab308">●</span> 7~8위 &nbsp;
  <span style="color:#22c55e">●</span> 9~10위
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False).add_to(m)

m.save('goyang_satellite_map.html')
print("완료! goyang_satellite_map.html 다운받아서 올려주세요!")

완료! goyang_satellite_map.html 다운받아서 올려주세요!


In [ ]:
!pip install folium scipy openpyxl -q

import pandas as pd
import numpy as np
from scipy.spatial import Voronoi
import folium
import json

# ── 1. 아파트 데이터 로드 ─────────────────────────────────────────────────────
# 다운받은 파일명으로 변경
apt = pd.read_excel('/content/20260320_단지_기본정보.xlsx', header=1)
print(f"전체 데이터: {len(apt):,}행")
print(apt.columns.tolist())

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


전체 데이터: 21,499행
['시도', '시군구', '읍면', '동리', '단지코드', '단지명', '단지분류', '법정동주소', '우편번호', '도로명주소', '분양형태', '사용승인일', '가입일', '동수', '세대수', '분양세대수', '임대세대수', '임대세대수(공공)', '임대세대수(민간)', '관리방식', '난방방식', '복도유형', '시공사', '시행사', '주택관리업자', '주택관리업자 사업자등록번호', '일반관리-관리방식', '일반관리-인원', '경비관리-관리방식', '경비관리-인원', '경비관리-계약업체', '청소관리-관리방식', '청소관리-인원', '청소관리-계약업체', '음식물 처리방법', '소독관리-관리방식', '소독관리-계약업체', '연간소독횟수', '소독방법', '건물구조', '전기-수전용량', '전기-세대전기계약방식', '전기안전관리자선임여부', '화재수신반방식', '급수방식', '승강기관리-관리방식', '승강기(승객용)', '승강기(화물용)', '승강기(승객+화물)', '승강기(장애인)', '승강기(비상용)', '승강기(기타)', '총주차대수', '지상주차대수', '지하주차대수', 'CCTV대수', '홈네트워크', '긴급자동차 출입가능한 출입문', '선택한출입문의 주차관제시스템', '긴급전용번호판자동인식여부', '긴급자동차 출입방법', '긴급자동차 즉시통과', '관리사무소 주소', '관리사무소 연락처', '관리사무소 팩스', '부대복리시설', '최고층수', '최고층수(건축물대장상)', '지하층수', '차량보유대수(전체)', '차량보유대수(전기차)', '전기충전기설치여부(지상)', '전기충전기설치여부(지하)', '전기차전용주차면수(지상)', '전기차전용주차면수(지하)', '외부인개방여부(지상)', '외부인개방여부(지하)', '이용가능시작시간(지상)', '이용가능시작시간(지하)', '이용종료시작시간(지상)', '이용종료시작시간(지하)', '전기차충전시설(상세)', '전기차 충전시설 설치대수(지상)', '전기차 충전시설 설치대수(지

In [ ]:
# ── 2. 고양시 필터링 ──────────────────────────────────────────────────────────
goyang_apt = apt[apt['시군구'].str.contains('고양', na=False)].copy()
print(f"고양시 아파트 단지 수: {len(goyang_apt)}개")
print(f"총 세대수: {goyang_apt['세대수'].sum():,}세대")
print(goyang_apt[['단지명','동리','세대수']].head(10).to_string())

고양시 아파트 단지 수: 450개
총 세대수: 289,275.0세대
                       단지명   동리     세대수
11139              고양원당줌시티  주교동   264.0
11140                대동피렌체  주교동   126.0
11141          세창짜임주상복합아파트  주교동   109.0
11142                 주교벽산  주교동   283.0
11143             주교우림루미아트  주교동   163.0
11144              주교우인아파트  주교동   299.0
11145  고양 삼송A 4블럭 신원마을 4단지  신원동  1302.0
11146             고양삼송아이파크  신원동   610.0
11147           고양삼송우남퍼스트빌  신원동   611.0
11148        고양삼송호반베르디움9단지  신원동   353.0


In [ ]:
!pip install geopy -q
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

# ── 3. 고양시 아파트 좌표 지오코딩 ───────────────────────────────────────────
geolocator = Nominatim(user_agent="goyang_apt_analysis")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# 동리 + 단지명으로 좌표 추정 (속도를 위해 동 중심 좌표 활용)
# 먼저 동별 중심 좌표를 편의점 데이터에서 가져오기
dong_coords = convenience2.groupby('행정동명').agg(
    lat=('위도', 'mean'),
    lng=('경도', 'mean')
).reset_index()
dong_coords['동리'] = dong_coords['행정동명'].str.strip()

# 아파트에 동 중심 좌표 매핑
goyang_apt['동리_clean'] = goyang_apt['동리'].str.strip()
goyang_apt_geo = goyang_apt.merge(
    dong_coords[['동리', 'lat', 'lng']],
    left_on='동리_clean', right_on='동리',
    how='left'
)

matched = goyang_apt_geo.dropna(subset=['lat', 'lng'])
unmatched = goyang_apt_geo[goyang_apt_geo['lat'].isna()]
print(f"좌표 매핑 성공: {len(matched)}개")
print(f"좌표 없음: {len(unmatched)}개")
print(f"\n매핑 안된 동 목록:")
print(unmatched['동리_clean'].unique())

좌표 매핑 성공: 82개
좌표 없음: 368개

매핑 안된 동 목록:
['신원동' '원흥동' '도내동' '성사동' '지축동' '오금동' '삼송동' '동산동' '토당동' '화정동' '행신동' '덕은동'
 '향동동' '중산동' '장항동' '마두동' '백석동' '풍동' '사리현동' '성석동' '일산동' '주엽동' '탄현동' '가좌동']


In [ ]:
# ── 세대수 상위 단지만 지오코딩 (300세대 이상) ───────────────────────────────
large_apt = goyang_apt[goyang_apt['세대수'] >= 300].copy().reset_index(drop=True)
print(f"300세대 이상 단지: {len(large_apt)}개 / 총 {large_apt['세대수'].sum():,.0f}세대")

# 지오코딩
geolocator = Nominatim(user_agent="goyang_apt_v2")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.2)

lats, lngs = [], []
for i, row in large_apt.iterrows():
    query = f"{row['도로명주소']}, 대한민국"
    try:
        loc = geocode(query)
        if loc:
            lats.append(loc.latitude)
            lngs.append(loc.longitude)
        else:
            lats.append(None)
            lngs.append(None)
    except:
        lats.append(None)
        lngs.append(None)
    if i % 20 == 0:
        print(f"진행중... {i}/{len(large_apt)}")

large_apt['lat'] = lats
large_apt['lng'] = lngs

success = large_apt.dropna(subset=['lat','lng'])
print(f"\n지오코딩 완료: {len(success)}개 / {len(large_apt)}개")
large_apt.to_csv('goyang_apt_geo.csv', index=False, encoding='utf-8-sig')
print("저장 완료!")

300세대 이상 단지: 345개 / 총 268,031세대
진행중... 0/345
진행중... 20/345


진행중... 40/345
진행중... 60/345
진행중... 80/345


진행중... 100/345
진행중... 120/345
진행중... 140/345
진행중... 160/345
진행중... 180/345
진행중... 200/345
진행중... 220/345


진행중... 240/345


진행중... 260/345


진행중... 280/345
진행중... 300/345
진행중... 320/345
진행중... 340/345

지오코딩 완료: 45개 / 345개
저장 완료!


In [ ]:
import requests
import time

KAKAO_API_KEY = "a4fc9dd67b9331192b6d4a03d31a87cd"

def kakao_geocode(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {"query": address}
    try:
        res = requests.get(url, headers=headers, params=params)
        docs = res.json().get('documents', [])
        if docs:
            return float(docs[0]['y']), float(docs[0]['x'])
    except:
        pass
    return None, None

# 전체 345개 재지오코딩
large_apt = pd.read_csv('goyang_apt_geo.csv')
lats, lngs = [], []

for i, row in large_apt.iterrows():
    lat, lng = kakao_geocode(row['도로명주소'])
    lats.append(lat)
    lngs.append(lng)
    if i % 30 == 0:
        print(f"진행중... {i}/{len(large_apt)}")
    time.sleep(0.1)  # 카카오는 빠름!

large_apt['lat'] = lats
large_apt['lng'] = lngs

success = large_apt.dropna(subset=['lat','lng'])
print(f"\n지오코딩 완료: {len(success)}개 / {len(large_apt)}개")
print(f"커버 세대수: {success['세대수'].sum():,.0f}세대")
large_apt.to_csv('goyang_apt_kakao.csv', index=False, encoding='utf-8-sig')
print("저장 완료!")

진행중... 0/345
진행중... 30/345
진행중... 60/345
진행중... 90/345
진행중... 120/345
진행중... 150/345
진행중... 180/345
진행중... 210/345
진행중... 240/345
진행중... 270/345
진행중... 300/345
진행중... 330/345

지오코딩 완료: 0개 / 345개
커버 세대수: 0세대
저장 완료!


In [ ]:
# API 키 테스트
import requests

KAKAO_API_KEY = "YOUR_ACTUAL_KAKAO_API_KEY_HERE"  # 실제 카카오 REST API 키를 여기에 입력하세요!

url = "https://dapi.kakao.com/v2/local/search/address.json"
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
params = {"query": "경기도 고양시 덕양구 권율대로 512"}

res = requests.get(url, headers=headers, params=params)
print("상태코드:", res.status_code)
print("응답:", res.json())

UnicodeEncodeError: 'latin-1' codec can't encode characters in position 8-10: ordinal not in range(256)

In [ ]:
# API 키 테스트
import requests

KAKAO_API_KEY = "a4fc9dd67b9331192b6d4a03d31a87cd"  # 키 다시 확인!

url = "https://dapi.kakao.com/v2/local/search/address.json"
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
params = {"query": "경기도 고양시 덕양구 권율대로 512"}

res = requests.get(url, headers=headers, params=params)
print("상태코드:", res.status_code)
print("응답:", res.json())

상태코드: 403
응답: {'errorType': 'NotAuthorizedError', 'message': 'App(지도) disabled OPEN_MAP_AND_LOCAL service.'}


In [ ]:
KAKAO_API_KEY = "a4fc9dd67b9331192b6d4a03d31a87cd"  # 여기만 바꾸면 됨

url = "https://dapi.kakao.com/v2/local/search/address.json"
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
params = {"query": "경기도 고양시 덕양구 권율대로 512"}

res = requests.get(url, headers=headers, params=params)
print("상태코드:", res.status_code)
print("응답:", res.json())

상태코드: 200
응답: {'documents': [{'address': {'address_name': '경기 고양시 덕양구 도내동 950', 'b_code': '4128110500', 'h_code': '4128153000', 'main_address_no': '950', 'mountain_yn': 'N', 'region_1depth_name': '경기', 'region_2depth_name': '고양시 덕양구', 'region_3depth_h_name': '흥도동', 'region_3depth_name': '도내동', 'sub_address_no': '', 'x': '126.870344396967', 'y': '37.6362669470587'}, 'address_name': '경기 고양시 덕양구 권율대로 512', 'address_type': 'ROAD_ADDR', 'road_address': {'address_name': '경기 고양시 덕양구 권율대로 512', 'building_name': '원흥퍼스트푸르지오시티', 'main_building_no': '512', 'region_1depth_name': '경기', 'region_2depth_name': '고양시 덕양구', 'region_3depth_name': '도내동', 'road_name': '권율대로', 'sub_building_no': '', 'underground_yn': 'N', 'x': '126.870344396967', 'y': '37.6362669470587', 'zone_no': '10551'}, 'x': '126.870344396967', 'y': '37.6362669470587'}], 'meta': {'is_end': True, 'pageable_count': 1, 'total_count': 1}}


In [ ]:
import requests, time
import pandas as pd

KAKAO_API_KEY = "a4fc9dd67b9331192b6d4a03d31a87cd"  # 아까 테스트한 키 그대로

def kakao_geocode(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {"query": address}
    try:
        res = requests.get(url, headers=headers, params=params)
        docs = res.json().get('documents', [])
        if docs:
            return float(docs[0]['y']), float(docs[0]['x'])
    except:
        pass
    return None, None

large_apt = pd.read_csv('goyang_apt_geo.csv')
lats, lngs = [], []

for i, row in large_apt.iterrows():
    lat, lng = kakao_geocode(str(row['도로명주소']))
    lats.append(lat)
    lngs.append(lng)
    if i % 50 == 0:
        print(f"{i}/{len(large_apt)} 완료...")
    time.sleep(0.05)

large_apt['lat'] = lats
large_apt['lng'] = lngs

success = large_apt.dropna(subset=['lat','lng'])
print(f"\n지오코딩 완료: {len(success)}개 / {len(large_apt)}개")
print(f"커버 세대수: {success['세대수'].sum():,.0f}세대")
large_apt.to_csv('goyang_apt_kakao.csv', index=False, encoding='utf-8-sig')
print("저장 완료!")

0/345 완료...
50/345 완료...
100/345 완료...
150/345 완료...
200/345 완료...
250/345 완료...
300/345 완료...

지오코딩 완료: 336개 / 345개
커버 세대수: 261,308세대
저장 완료!


In [ ]:
!pip install folium scipy -q

import pandas as pd
import numpy as np
import folium
import json
from scipy.spatial import Voronoi

# ── 데이터 로드 ───────────────────────────────────────────────────────────────
apt = pd.read_csv('goyang_apt_kakao.csv').dropna(subset=['lat','lng'])
apt = apt[
    (apt['lat'].between(37.57, 37.73)) &
    (apt['lng'].between(126.74, 126.97))
].copy()

stores = pd.read_csv('goyang_convenience_stores.csv')

print(f"아파트: {len(apt)}개 단지 / {apt['세대수'].sum():,.0f}세대")
print(f"편의점: {len(stores)}개")

# ── 보로노이 계산 ─────────────────────────────────────────────────────────────
store_coords = stores[['lat','lng']].values

# 경계 밖 더미 포인트 추가 (보로노이가 무한대로 뻗는 것 방지)
bbox_pts = np.array([
    [37.50, 126.65], [37.50, 127.05],
    [37.80, 126.65], [37.80, 127.05],
    [37.50, 126.85], [37.80, 126.85],
    [37.65, 126.65], [37.65, 127.05],
])
all_coords = np.vstack([store_coords, bbox_pts])
vor = Voronoi(all_coords)

# ── 각 편의점 보로노이 셀에 아파트 세대수 합산 ────────────────────────────────
from scipy.spatial import cKDTree
tree = cKDTree(store_coords)

# 각 아파트를 가장 가까운 편의점에 할당
apt_coords = apt[['lat','lng']].values
_, store_idx = tree.query(apt_coords)
apt['store_idx'] = store_idx
apt['store_name'] = apt['store_idx'].apply(lambda i: stores.iloc[i]['name'])
apt['store_brand'] = apt['store_idx'].apply(lambda i: stores.iloc[i]['brand'])

# 편의점별 담당 세대수 합산
store_coverage = apt.groupby('store_idx').agg(
    total_units=('세대수', 'sum'),
    apt_count=('단지명', 'count')
).reset_index()

stores_with_cov = stores.copy().reset_index()
stores_with_cov = stores_with_cov.merge(store_coverage, left_on='index', right_on='store_idx', how='left')
stores_with_cov['total_units'] = stores_with_cov['total_units'].fillna(0)

print(f"\n편의점당 평균 담당 세대수: {stores_with_cov['total_units'].mean():,.0f}세대")
print(f"최대 담당 세대수: {stores_with_cov['total_units'].max():,.0f}세대")
print(f"담당 세대수 0인 편의점 (주거지역 외): {(stores_with_cov['total_units']==0).sum()}개")
print("\n세대수 많이 담당하는 편의점 TOP 10:")
print(stores_with_cov.nlargest(10,'total_units')[['name','brand','dong','total_units','apt_count']].to_string(index=False))

아파트: 325개 단지 / 252,967세대
편의점: 951개

편의점당 평균 담당 세대수: 266세대
최대 담당 세대수: 4,320세대
담당 세대수 0인 편의점 (주거지역 외): 711개

세대수 많이 담당하는 편의점 TOP 10:
         name brand dong  total_units  apt_count
    세븐일레븐일산동구 세븐일레븐 백석1동       4320.0        1.0
     지에스25뉴식사  GS25  식사동       3219.0        2.0
        아이스크림    기타 화정1동       3188.0        3.0
이마트24고양연합생활관점 이마트24  흥도동       2981.0        3.0
     지에스25탄현큰  GS25 탄현2동       2938.0        2.0
      세븐행신샘터점 세븐일레븐 행신1동       2920.0        2.0
    씨유삼송신원2단지    CU  원신동       2889.0        3.0
   씨유덕양어울림누리점    CU 성사2동       2782.0        2.0
    GS25일산제니스  GS25 탄현1동       2700.0        1.0
    씨유일산에듀포레점    CU 탄현2동       2595.0        2.0


In [ ]:
!pip install folium scipy -q

import pandas as pd
import numpy as np
import folium
import json
from scipy.spatial import Voronoi

# ── 데이터 로드 ───────────────────────────────────────────────────────────────
apt = pd.read_csv('goyang_apt_kakao.csv').dropna(subset=['lat','lng'])
apt = apt[
    (apt['lat'].between(37.57, 37.73)) &
    (apt['lng'].between(126.74, 126.97))
].copy()

stores = pd.read_csv('goyang_convenience_stores.csv')
store_coords = stores[['lat','lng']].values

# ── 편의점별 담당 세대수 계산 ─────────────────────────────────────────────────
from scipy.spatial import cKDTree
tree = cKDTree(store_coords)
apt_coords = apt[['lat','lng']].values
_, store_idx = tree.query(apt_coords)
apt['store_idx'] = store_idx

store_coverage = apt.groupby('store_idx').agg(
    total_units=('세대수', 'sum'),
    apt_count=('단지명', 'count')
).reset_index()

stores_r = stores.copy().reset_index()
stores_r = stores_r.merge(store_coverage, left_on='index', right_on='store_idx', how='left')
stores_r['total_units'] = stores_r['total_units'].fillna(0).astype(int)
stores_r['apt_count']   = stores_r['apt_count'].fillna(0).astype(int)

# ── 보로노이 다이어그램 계산 ──────────────────────────────────────────────────
# 위경도 → 평면 좌표 변환 (미터 단위)
def to_xy(lat, lng, lat0=37.65, lng0=126.83):
    x = (lng - lng0) * 88000
    y = (lat - lat0) * 111000
    return x, y

def to_latlng(x, y, lat0=37.65, lng0=126.83):
    return lat0 + y/111000, lng0 + x/88000

# 편의점 XY 좌표
pts_xy = np.array([to_xy(r['lat'], r['lng']) for _, r in stores_r.iterrows()])

# 경계 더미 포인트
dummy = np.array([
    [-15000,-12000],[-15000, 12000],[15000,-12000],[15000,12000],
    [-15000, 0],[15000, 0],[0,-12000],[0,12000],
    [-10000,-12000],[10000,-12000],[-10000,12000],[10000,12000],
])
all_pts = np.vstack([pts_xy, dummy])
vor = Voronoi(all_pts)

# 고양시 클립 경계
clip_lat = [37.57, 37.57, 37.73, 37.73, 37.57]
clip_lng = [126.74, 126.97, 126.97, 126.74, 126.74]

def clip_polygon(poly, bounds):
    from shapely.geometry import Polygon
    try:
        p = Polygon(poly)
        b = Polygon(list(zip(bounds[1], bounds[0])))
        clipped = p.intersection(b)
        if clipped.is_empty: return None
        if clipped.geom_type == 'Polygon':
            return list(clipped.exterior.coords)
        return None
    except:
        return None

# ── 세대수 기준 색상 계산 ─────────────────────────────────────────────────────
max_units = stores_r['total_units'].max()

def units_to_color(units):
    if units == 0:
        return '#cccccc', 0.15
    ratio = min(units / 2000, 1.0)
    if ratio < 0.33:
        r,g,b = 65,182,196
    elif ratio < 0.66:
        r,g,b = 253,141,60
    else:
        r,g,b = 189,0,38
    return f'#{r:02x}{g:02x}{b:02x}', 0.35 + ratio*0.3

# ── 지도 생성 ─────────────────────────────────────────────────────────────────
m = folium.Map(location=[37.65, 126.83], zoom_start=12,
               tiles='CartoDB positron')

# 보로노이 셀 그리기
!pip install shapely -q
from shapely.geometry import Polygon as ShapelyPolygon

clip_box = ShapelyPolygon([
    (126.74,37.57),(126.97,37.57),(126.97,37.73),(126.74,37.73)
])

vor_group = folium.FeatureGroup(name='보로노이 (편의점 담당구역)', show=True)

for i, region_idx in enumerate(vor.point_region[:len(stores_r)]):
    region = vor.regions[region_idx]
    if -1 in region or len(region) == 0:
        continue
    try:
        poly_xy = vor.vertices[region]
        poly_latlng = [to_latlng(x, y) for x,y in poly_xy]
        poly_shapely = ShapelyPolygon([(lng, lat) for lat,lng in poly_latlng])
        clipped = poly_shapely.intersection(clip_box)
        if clipped.is_empty:
            continue
        if clipped.geom_type == 'Polygon':
            coords = [[lat, lng] for lng, lat in clipped.exterior.coords]
        elif clipped.geom_type == 'MultiPolygon':
            coords = [[lat, lng] for lng, lat in list(clipped.geoms[0].exterior.coords)]
        else:
            continue

        row = stores_r.iloc[i]
        color, opacity = units_to_color(row['total_units'])

        folium.Polygon(
            locations=coords,
            color='white', weight=0.5, opacity=0.6,
            fill=True, fill_color=color, fill_opacity=opacity,
            tooltip=f"{row['name']} ({row['brand']})<br>담당 세대수: {int(row['total_units']):,}세대",
            popup=folium.Popup(
                f"<b>{row['name']}</b><br>"
                f"브랜드: {row['brand']}<br>"
                f"담당 아파트: {int(row['apt_count'])}단지<br>"
                f"담당 세대수: {int(row['total_units']):,}세대",
                max_width=220
            )
        ).add_to(vor_group)
    except:
        continue

vor_group.add_to(m)

# 편의점 마커
brand_colors = {'CU':'#00a0e9','GS25':'#009d4e','세븐일레븐':'#ff6600','이마트24':'#e11400','기타':'#aaaaaa'}
store_group = folium.FeatureGroup(name='편의점', show=True)
for _, row in stores_r.iterrows():
    color = brand_colors.get(row['brand'], '#aaaaaa')
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius=3, color=color, fill=True,
        fill_color=color, fill_opacity=0.9,
        tooltip=f"{row['name']} ({row['brand']})"
    ).add_to(store_group)
store_group.add_to(m)

# 아파트 마커
apt_group = folium.FeatureGroup(name='아파트 단지', show=False)
for _, row in apt.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius=max(3, min(10, int(row['세대수'])//300)),
        color='#7c3aed', fill=True,
        fill_color='#7c3aed', fill_opacity=0.6,
        tooltip=f"{row['단지명']} {int(row['세대수']):,}세대"
    ).add_to(apt_group)
apt_group.add_to(m)

# 부하 높은 편의점 TOP 10 강조
top_burden = stores_r.nlargest(10, 'total_units')
burden_group = folium.FeatureGroup(name='과부하 편의점 TOP 10', show=True)
for rank, (_, row) in enumerate(top_burden.iterrows(), 1):
    folium.Marker(
        location=[row['lat'], row['lng']],
        icon=folium.DivIcon(
            html=f"""<div style="
                width:26px;height:26px;border-radius:50%;
                background:#bd0026;border:2px solid white;
                display:flex;align-items:center;justify-content:center;
                font-size:11px;font-weight:700;color:white;
                box-shadow:0 2px 6px rgba(0,0,0,0.4);
                line-height:26px;text-align:center;">{rank}</div>""",
            icon_size=(26,26), icon_anchor=(13,13)
        ),
        tooltip=f"과부하 #{rank} | {row['name']} | {int(row['total_units']):,}세대",
        popup=folium.Popup(
            f"<b>과부하 #{rank}: {row['name']}</b><br>"
            f"브랜드: {row['brand']}<br>"
            f"담당 세대수: {int(row['total_units']):,}세대<br>"
            f"담당 단지 수: {int(row['apt_count'])}개",
            max_width=220
        )
    ).add_to(burden_group)
burden_group.add_to(m)

# 범례
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:14px 18px;border-radius:10px;
     border:1px solid #ddd;font-size:12px;line-height:2;
     box-shadow:0 2px 8px rgba(0,0,0,0.15)">
  <b style="font-size:13px">고양시 편의점 보로노이 분석</b><br>
  <b style="font-size:11px;color:#888">담당 세대수</b><br>
  <span style="color:#41b6c4">■</span> ~1,000세대 &nbsp;
  <span style="color:#fd8d3c">■</span> ~2,000세대 &nbsp;
  <span style="color:#bd0026">■</span> 2,000세대+<br>
  <span style="color:#cccccc">■</span> 주거지역 외<br>
  <span style="color:#bd0026">●</span> 과부하 TOP 10
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False).add_to(m)

m.save('goyang_voronoi_map.html')
print("완료! goyang_voronoi_map.html 다운받아서 올려주세요!")

완료! goyang_voronoi_map.html 다운받아서 올려주세요!


In [ ]:
index_html = """<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>고양시 편의점 입지 분석</title>
<style>
  * { margin:0; padding:0; box-sizing:border-box; }
  body { font-family: system-ui, sans-serif; background:#f4f6f8; color:#1a1f2e; }
  header {
    background:white; border-bottom:1px solid #e5e7eb;
    padding:20px 40px; display:flex; align-items:center; justify-content:space-between;
  }
  .logo { display:flex; align-items:center; gap:10px; }
  .dot { width:10px; height:10px; border-radius:50%; background:#00a0e9;
         animation:pulse 2s infinite; }
  @keyframes pulse { 0%,100%{opacity:1}50%{opacity:0.4} }
  h1 { font-size:18px; font-weight:600; }
  .sub { font-size:12px; color:#6b7280; margin-top:2px; }
  .tags { display:flex; gap:6px; }
  .tag { font-size:11px; padding:3px 10px; border-radius:20px;
         border:1px solid #e5e7eb; color:#6b7280; }
  main { max-width:1100px; margin:0 auto; padding:40px 24px; }
  .stats { display:grid; grid-template-columns:repeat(4,1fr); gap:12px; margin-bottom:36px; }
  .stat { background:white; border-radius:12px; padding:16px 20px;
          border:1px solid #e5e7eb; }
  .stat-label { font-size:12px; color:#6b7280; margin-bottom:4px; }
  .stat-value { font-size:24px; font-weight:600; }
  .cards { display:grid; grid-template-columns:repeat(3,1fr); gap:16px; }
  .card {
    background:white; border-radius:14px; border:1px solid #e5e7eb;
    overflow:hidden; text-decoration:none; color:inherit;
    transition:transform 0.15s, box-shadow 0.15s;
  }
  .card:hover { transform:translateY(-3px); box-shadow:0 8px 24px rgba(0,0,0,0.1); }
  .card-thumb {
    height:180px; display:flex; align-items:center; justify-content:center;
    font-size:48px;
  }
  .card-body { padding:16px 20px; }
  .card-title { font-size:15px; font-weight:600; margin-bottom:6px; }
  .card-desc { font-size:13px; color:#6b7280; line-height:1.6; margin-bottom:12px; }
  .card-badge { display:inline-block; font-size:11px; padding:2px 8px;
                border-radius:10px; }
  footer { text-align:center; padding:32px; font-size:12px; color:#9ca3af; }
</style>
</head>
<body>

<header>
  <div class="logo">
    <div class="dot"></div>
    <div>
      <div style="font-size:16px;font-weight:600">고양시 편의점 입지 분석</div>
      <div class="sub">소상공인 상가 데이터 · 공동주택 세대수 · K-means · 보로노이</div>
    </div>
  </div>
  <div class="tags">
    <span class="tag">Python</span>
    <span class="tag">Folium</span>
    <span class="tag">Kakao API</span>
    <span class="tag">Scipy</span>
  </div>
</header>

<main>
  <div class="stats">
    <div class="stat">
      <div class="stat-label">분석 대상 편의점</div>
      <div class="stat-value" style="color:#00a0e9">951<span style="font-size:14px;font-weight:400;color:#6b7280">개</span></div>
    </div>
    <div class="stat">
      <div class="stat-label">아파트 단지</div>
      <div class="stat-value" style="color:#7c3aed">325<span style="font-size:14px;font-weight:400;color:#6b7280">개</span></div>
    </div>
    <div class="stat">
      <div class="stat-label">커버 세대수</div>
      <div class="stat-value" style="color:#059669">25.3<span style="font-size:14px;font-weight:400;color:#6b7280">만</span></div>
    </div>
    <div class="stat">
      <div class="stat-label">공백 심각 동네</div>
      <div class="stat-value" style="color:#dc2626">10<span style="font-size:14px;font-weight:400;color:#6b7280">곳</span></div>
    </div>
  </div>

  <div class="cards">
    <a class="card" href="goyang_final_map.html">
      <div class="card-thumb" style="background:#eff6ff">🗺️</div>
      <div class="card-body">
        <div class="card-title">공백 지역 분석</div>
        <div class="card-desc">편의점 간 거리 기반으로 공백이 심각한 행정동 TOP 10을 도출한 지도</div>
        <span class="card-badge" style="background:#eff6ff;color:#1d4ed8">공백 분석</span>
      </div>
    </a>
    <a class="card" href="goyang_satellite_map.html">
      <div class="card-thumb" style="background:#f0fdf4">🛰️</div>
      <div class="card-body">
        <div class="card-title">위성지도 뷰</div>
        <div class="card-desc">위성사진 위에 편의점과 추천 부지를 오버레이한 지도. 실제 지형 확인 가능</div>
        <span class="card-badge" style="background:#f0fdf4;color:#166534">위성지도</span>
      </div>
    </a>
    <a class="card" href="goyang_voronoi_map.html">
      <div class="card-thumb" style="background:#fdf4ff">📐</div>
      <div class="card-body">
        <div class="card-title">보로노이 세대수 분석</div>
        <div class="card-desc">각 편의점의 담당 구역을 보로노이로 나눠 세대수 부하를 시각화한 지도</div>
        <span class="card-badge" style="background:#fdf4ff;color:#7e22ce">보로노이</span>
      </div>
    </a>
  </div>
</main>

<footer>고양시 편의점 입지 분석 · 데이터 출처: 소상공인시장진흥공단, 국토교통부, 카카오 지도 API</footer>

</body>
</html>"""

with open('index.html', 'w', encoding='utf-8') as f:
    f.write(index_html)
print("index.html 생성 완료!")

index.html 생성 완료!
